In [1]:
!pip install selenium beautifulsoup4 pandas
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import WebDriverException
import time
import pandas as pd
from bs4 import BeautifulSoup
import os



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ✅ Set up WebDriver (Make sure chromedriver.exe is in the same folder)
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

# ✅ LinkedIn Login
try:
    driver.get("https://www.linkedin.com/login")
    time.sleep(3)  # ✅ Wait for page to load

    # Enter credentials (Change with your LinkedIn credentials)
    username = "gmail.com"  # ⬅️ Replace
    password = ""  # ⬅️ Replace

    email_field = driver.find_element(By.ID, "username")
    email_field.send_keys(username)

    password_field = driver.find_element(By.ID, "password")
    password_field.send_keys(password)
    password_field.send_keys(Keys.RETURN)

    time.sleep(5)  # ✅ Wait for login to complete
    print("✅ Logged in to LinkedIn")

except WebDriverException as e:
    print(f"❌ Error logging in: {e}")


✅ Logged in to LinkedIn


In [3]:
# ✅ Open UNT Alumni search page
search_url = "https://www.linkedin.com/school/university-of-north-texas/people/"
driver.get(search_url)
time.sleep(5)  # ✅ Wait for results to load

# ✅ Scroll multiple times to load more results
for _ in range(10):  # Scroll 10 times to load more alumni
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)  # ✅ Wait longer for new profiles to load

print("✅ Scrolled through UNT Alumni page")


✅ Scrolled through UNT Alumni page


In [4]:
# ✅ Extract LinkedIn profile links
soup = BeautifulSoup(driver.page_source, "html.parser")

profile_links = set()  # ✅ Use a set to store unique profile links
for link in soup.find_all('a', href=True):
    href = link['href']
    if "linkedin.com/in/" in href:
        full_url = href if href.startswith("http") else f"https://www.linkedin.com{href}"
        profile_links.add(full_url)

print(f"✅ Found {len(profile_links)} unique alumni profiles")
print(list(profile_links)[:5])  # ✅ Show first 5 profiles


✅ Found 39 unique alumni profiles
['https://www.linkedin.com/in/zachary-norris?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAACcmozcB0y03h2-EkZZrq7fkv5uG453rvLw', 'https://www.linkedin.com/in/zachary-warren-52v2142238?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAADtCKFsBXYxE_zDiuloSuXZ7CqX3r9rPxp4', 'https://www.linkedin.com/in/edward-asante-unt1?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAADpVkYQBf8hAXPvMVjfLrALf5GO4sm0bhOU', 'https://www.linkedin.com/in/beatrizgutierrez7?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAAEE3JZYBPY1rpFiLfPPTMsBvWfLVh55Yz6M', 'https://www.linkedin.com/in/davieonjackson?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAAC-X3QYBjXTQfi92ElucPdXd8-T31Q-YqN8']


In [5]:
# ✅ Load existing CSV to avoid duplicates
csv_filename = "UNT_Alumni_Data.csv"
if os.path.exists(csv_filename):
    existing_data = pd.read_csv(csv_filename)
    existing_profiles = set(existing_data['profile_url'])
else:
    existing_data = pd.DataFrame(columns=['name', 'headline', 'location', 'status', 'major', 'graduation_year', 'department', 'profile_url'])
    existing_profiles = set()

print(f"✅ Existing profiles loaded: {len(existing_profiles)}")


✅ Existing profiles loaded: 140


In [6]:
def scrape_profile(profile_url):
    if profile_url in existing_profiles:
        print(f"⏩ Skipping (already saved): {profile_url}")
        return None

    # ✅ Retry logic for network issues
    for attempt in range(3):
        try:
            driver.get(profile_url)
            time.sleep(5)
            break
        except WebDriverException:
            if attempt == 2:
                print(f"❌ Skipping {profile_url} (Network Error)")
                return None
            print(f"🔄 Retrying... {profile_url}")
            time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # ✅ Extract Name
    name_tag = soup.find('h1')
    name = name_tag.get_text().strip() if name_tag else "Not Found"

    # ✅ Extract Job Title (Alumni vs Faculty)
    headline_tag = soup.find('div', {'class': 'text-body-medium'})
    headline = headline_tag.get_text().strip() if headline_tag else "Not Found"
    
    # ✅ Determine if this is an **Alumni or Faculty**
    status = "Faculty" if "professor" in headline.lower() or "faculty" in headline.lower() else "Alumni"

    # ✅ Extract Correct Location
    location = "Not Found"
    location_tag = soup.find('span', {'class': 'text-body-small'})
    if location_tag:
        location = location_tag.get_text().strip()
        # Remove unnecessary text like "2nd degree connection"
        location = location.replace("2nd degree connection", "").strip()
        location = location.replace("(He/Him)", "").replace("(She/Her)", "").strip()

    # ✅ Extract Education Details (Major, Department, and Graduation Year)
    major = "Not Found"
    graduation_year = "Not Found"
    department = "Not Found"

    education_section = soup.find('section', {'id': 'education-section'})
    if education_section:
        edu_entries = education_section.find_all('div', {'class': 'pv-entity__summary-info'})
        for edu in edu_entries:
            school = edu.find('h3')
            if school and "north texas" in school.get_text().lower():
                # ✅ Extract Major
                major_tag = edu.find('p', {'class': 'pv-entity__fos'})
                major = major_tag.get_text().strip() if major_tag else "Not Found"

                # ✅ Extract Department
                department_tag = edu.find_all('p')[1] if len(edu.find_all('p')) > 1 else None
                department = department_tag.get_text().strip() if department_tag else "Not Found"

                # ✅ Extract Graduation Year
                date_range = edu.find('span', {'class': 'pv-entity__dates'})
                graduation_year = date_range.get_text().strip() if date_range else "Not Found"

                break  # ✅ Stop after finding first UNT education entry

    return {
        'name': name,
        'headline': headline,
        'location': location,
        'status': status,
        'major': major,
        'graduation_year': graduation_year,
        'department': department,
        'profile_url': profile_url
    }

print("✅ Profile scraping function updated and ready")


✅ Profile scraping function updated and ready


In [7]:
# ✅ Scrape multiple profiles and add new ones
all_profiles = []
for url in profile_links:
    if url in existing_profiles:
        print(f"⏩ Skipping (already saved): {url}")
        continue

    profile_data = scrape_profile(url)
    if profile_data:
        all_profiles.append(profile_data)
        print(f"✅ Scraped: {profile_data['name']}")

# ✅ Merge with existing data
if all_profiles:
    new_df = pd.DataFrame(all_profiles)
    updated_df = pd.concat([existing_data, new_df], ignore_index=True)
    updated_df.to_csv(csv_filename, index=False)
    print(f"✅ {len(new_df)} new profiles added. Total: {len(updated_df)} profiles saved.")
else:
    print("✅ No new profiles found. Everything is already saved.")


⏩ Skipping (already saved): https://www.linkedin.com/in/zachary-norris?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAACcmozcB0y03h2-EkZZrq7fkv5uG453rvLw
⏩ Skipping (already saved): https://www.linkedin.com/in/zachary-warren-52v2142238?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAADtCKFsBXYxE_zDiuloSuXZ7CqX3r9rPxp4
⏩ Skipping (already saved): https://www.linkedin.com/in/edward-asante-unt1?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAADpVkYQBf8hAXPvMVjfLrALf5GO4sm0bhOU
⏩ Skipping (already saved): https://www.linkedin.com/in/beatrizgutierrez7?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAAEE3JZYBPY1rpFiLfPPTMsBvWfLVh55Yz6M
⏩ Skipping (already saved): https://www.linkedin.com/in/davieonjackson?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAAC-X3QYBjXTQfi92ElucPdXd8-T31Q-YqN8


✅ Scraped: Foye (Aishat) Oyedeji
⏩ Skipping (already saved): https://www.linkedin.com/in/siva-naga-jyothi-m-2aa6b126b?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAAEIh-2IBv_9dsayjwm-bqOrPpXYfpSks8pU
✅ Scraped: Ulligadda Sreeja
⏩ Skipping (already saved): https://www.linkedin.com/in/terrancep?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAADWMfxoB31S6sGVLrEFZAbpxyTVmYSR3axA
⏩ Skipping (already saved): https://www.linkedin.com/in/hassan-qandil-phd-pe-cem%C2%AE-a65325a0
⏩ Skipping (already saved): https://www.linkedin.com/in/kirubel-alemu-b73ba31b5?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAADIwGMcBWyOqK-NQEOwhsvSucSOO2ygkaO0
⏩ Skipping (already saved): https://www.linkedin.com/in/siripurapuindiradevi?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAACjQmUEBzC1jl1HzU8KVQmiq5n7gYAn1yn0
⏩ Skipping (already saved): https://www.linkedin.com/in/seema-aarella?miniProfileUrn=urn%3Ali%3Afs_miniProfile%3AACoAACKdObIB4WVXj6QXu1Gp9Jkkz8HCRhte_U4
✅ Scraped: Arrin B.
✅ Scraped: Huyen Nguyen
⏩ Skip

In [8]:
driver.quit()
print("✅ WebDriver closed")


✅ WebDriver closed
